# Part 1: Linear Regression on the Tips Dataset

To begin, let's load the tips dataset from the `seaborn` library.  This dataset contains records of tips, total bill, and information about the person who paid the bill. **We'll be trying to predict tips from the other data.**

## Data Loading and Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.feature_extraction import DictVectorizer
import matplotlib.pyplot as plt
np.random.seed(42)
plt.style.use('fivethirtyeight')
sns.set()
sns.set_context("talk")
%matplotlib inline

In [ ]:
data = sns.load_dataset("tips")

print("Number of Records:", len(data))
data.head()

In [ ]:
y = data['tip']
X = data.drop(columns='tip')

We do some feature engineering to convert the categorical columns into one-hot coding.

In [ ]:
def one_hot_encode_revised(data):
    """
    Return the one-hot encoded dataframe of our input data, removing redundancies.

    Parameters
    -----------
    data: a dataframe that may include non-numerical features

    Returns
    -----------
    A one-hot encoded dataframe that only contains numeric features without any redundancies.

    """
    columns = ['sex', 'smoker', 'day', 'time']
    for column in columns:
        values = data[column].unique()
        for value in values[:-1]:
            data[column + '=' + value] = (data[column] == value).astype(int)
        data = data.drop(column, axis=1)
    return data

In [ ]:
X = one_hot_encode_revised(X)

In [ ]:
X.head()

**Insert code below to perform a train/test split on X and Y, with 80/20 ratio**

## Linear regression with the model from Sklearn

We start with using the linear regression model from sklearn library

**Insert code below to create a linear regression model, train the model on the training set**

In [ ]:
from sklearn.linear_model import LinearRegression

...

**Insert code below to make a prediction on the test dataset, and calculate the mean square error**

The mean square error should be around 0.7 if your implementation is correct.

## Linear regression with self-defined model

In [ ]:
X.head()

Now that all of our data is numeric, we can begin to define our model function. Notice that after one-hot encoding our data, we now have 8 features instead of 6. Therefore, our linear model now looks like:

$$ \text{Tip} = \theta_1 \cdot \text{total}\_\text{bill} + \theta_2 \cdot \text{billsize} +...  $$

We can represent the linear combination above as a matrix-vector product.

Below, we create a `MyLinearModel` class with two methods, `predict` and `fit`.

The `predict` method gives a prediction based on the linear model:
$$ \text{prediction} = \theta \cdot X^{T} $$,
where $\theta = [\theta_1, \theta_2, ...]$, $X = [X_1, X_2, ...]$.

The `fit` method should train the model (i.e., calculating the best parameters $\theta$ that minimizes the error). But for now we just set all of its parameters to zero.

In [ ]:
class MyLinearModel():
    def predict(self, X):
        return X @ self._thetas

    def fit(self, X, y):
        number_of_features = X.shape[1]
        self._thetas = np.zeros(shape = (number_of_features, 1))

Here is an example to creating a linear model, fit the model.  
Since we only set zeros to theta in the `fit` method, `model._thetas` gives all zeros.

In [ ]:
model = MyLinearModel()
model.fit(X_train, y_train)

print("model parameters:\n", model._thetas)

**Insert code below, check the mean square error of this model on both training set, and test set**

## Fitting a Linear Model using scipy.optimize.minimize Methods

We will use the `scipy.optimize.minimize` function, to implement the fit method of the linear model.  
Note that we've added a `loss_function` parameter where the model is fit using the desired loss function.

In [ ]:
from scipy.optimize import minimize

def l1(y, y_hat):
    return np.abs(y - y_hat)

def l2(y, y_hat):
    return (y - y_hat)**2

class MyLinearModel():
    def predict(self, X):
        return X @ self._thetas

    def fit(self, loss_function, X, y):
        """
        Produce the estimated optimal _thetas for the given loss function,
        feature matrix X, and observations y.

        Parameters
        -----------
        loss_function: either the squared or absolute loss functions defined above
        X: a 2D dataframe (or numpy array) of numeric features (one-hot encoded)
        y: a 1D vector of tip amounts

        Returns
        -----------
        The estimate for the optimal theta vector that minimizes our loss
        """

        # The code starts with an initial guess (starting_guess) of all zeros for theta.
        # It then computes the average loss using the given loss_function (either l1 or l2).
        # The minimize function adjusts the theta values to reduce the loss.
        # This process is akin to gradient descent but handled internally by minimize,
        # which uses more advanced optimization techniques (e.g., BFGS) to find the minimum.
        number_of_features = X.shape[1]

        starting_guess = np.zeros(number_of_features)
        self._thetas = minimize(lambda theta:
                                np.mean(loss_function(y,X@theta))
                                , x0 = starting_guess)['x']
        # Notice above that we extract the 'x' entry in the dictionary returned by `minimize`.
        # This entry corresponds to the optimal theta estimated by the function. Sorry
        # we know it's a little confusing, but 'x' is hard coded into the minimize function
        # because of the fact that in the optimization universe "x" is what you optimize over.
        # It'd be less confusing if they used "theta".



Now, we create a new linear model, fit the model with `l2` loss, and show the trained parameters $\theta$.

In [ ]:
# When you run the code below, you should get back some non zero thetas.

model = MyLinearModel()
model.fit(l2, X_train, y_train)
model._thetas

**Insert code below, print the mean square error of this model on both training set, and test set**

## Implementing `fit` Function with Gradient Descent

To modify the `fit` function to use gradient descent instead of `scipy.optimize.minimize`, we need to update the theta parameters iteratively by computing the gradient of the loss function and updating the parameters in small steps (i.e., a small learning rate).

In [ ]:
import numpy as np

class MyLinearModel():
    def predict(self, X):
        return X @ self._thetas

    def fit(self, loss_function, X, y, learning_rate=0.001, iterations=100):
        """
        Fit the model using gradient descent to minimize the loss function.

        Parameters
        -----------
        loss_function: either the squared loss (l2) or absolute loss (l1) function.
        X: a 2D numpy array of numeric features.
        y: a 1D numpy array of the target values (regression targets).
        learning_rate: step size for gradient descent.
        iterations: number of gradient descent iterations to run.

        Returns
        -----------
        Optimal theta values after performing gradient descent.
        """

        # Initialize theta parameters to zeros
        number_of_features = X.shape[1]
        self._thetas = np.zeros(number_of_features)

        # Iterate to perform gradient descent
        m = len(y)  # Number of training examples

        for i in range(iterations):
            predictions = self.predict(X)  # Compute predictions: X @ theta

            # Calculate the gradient of the loss
            if loss_function == l2:
                # For squared loss (l2)
                errors = predictions - y
                gradient = (2/m) * X.T @ errors
            elif loss_function == l1:
                # For absolute loss (l1)
                errors = np.sign(predictions - y)
                gradient = (1/m) * X.T @ errors

            # Update the parameters using the gradient
            self._thetas -= learning_rate * gradient

        return self._thetas


In [ ]:
# When you run the code below, you should get back some non zero thetas.

model = MyLinearModel()
model.fit(l2, X_train, y_train, learning_rate=0.0005, iterations=50)
model._thetas

**Insert code cell below to calculate the mean square error of the model on test set**

**Insert code cell below, train a linear model with `l1` loss, print out the mean square error on test set**

# Part 2: Gradient Descent on Sinusoidal Data

Load the csv "lab8_data" from Google Drive

In [ ]:
# Load the Drive helper and mount
from google.colab import drive

# Prompt for Authorization
drive.mount('/content/drive')

In [ ]:
!ls /content/drive/MyDrive/lab8/lab8_data.csv

In [ ]:
# Run this cell to load the data for this problem
df = pd.read_csv("/content/drive/MyDrive/lab8/lab8_data.csv", index_col=0)
df.head()

This data has only two columns - `x` and `y`.  
If we plot this data, we see that there is a clear sinusoidal relationship between x and y.

In [ ]:
import plotly.express as px
px.scatter(df, x = "x", y = "y")

In this exercise, we'll show gradient descent is so powerful it can even optimize a nonlinear model. Specifically, we're going to model the relationship of our data by:
$$\Large{
f_{\boldsymbol{\theta(x)}} = \theta_1x + sin(\theta_2x)
}$$

Our model is parameterized by both $\theta_1$ and $\theta_2$, which we can represent in the vector, $\boldsymbol{\theta}$.

Note that a general sine function $a\sin(bx+c)$ has three parameters: amplitude scaling parameter $a$, frequency parameter $b$ and phase shifting parameter $c$.

Here, we're assuming the amplitude $a$ is around 1, and the phase shifting parameter $c$ is around zero. We do not attempt to justify this assumption and you're welcome to see what happens if you ignore this assumption.

You might ask why we don't just create a linear model like we did earlier with a sinusoidal feature. The issue is that the theta is <mark>INSIDE</mark> the sin function. In other words, linear models use their parameters to adjust the scale of each feature, but $\theta_2$ in this model adjusts the frequency of the feature. There are tricks we could play to use our linear model framework here, but we won't attempt this in our lab.

We define the `sin_model` function below that predicts $\textbf{y}$ (the $y$-values) using $\textbf{x}$ (the $x$-values) based on our new equation.

In [ ]:
def sin_model(x, theta):
    """
    Predict the estimate of y given x, theta_1, theta_2

    Keyword arguments:
    x -- the vector of values x
    theta -- a vector of length 2, where theta[0] = theta_1 and theta[1] = theta_2
    """
    theta_1 = theta[0]
    theta_2 = theta[1]
    return theta_1 * x + np.sin(theta_2 * x)

Recall $\hat{\theta}$ is the value of $\theta$ that minimizes our loss function. One way of solving for $\hat{\theta}$ is by computing the gradient of our loss function with respect to $\theta$. Recall that the gradient is a column vector of two partial derivatives.

Below are the expressions for calculating the gradients. Use them to fill in the functions below.

$L(\textbf{x}, \textbf{y}, \theta_1, \theta_2)$: our loss function, the mean squared error

$L(\textbf{x}, \textbf{y}, \theta_1, \theta_2)
= \frac{1}{n}\sum_{i=1}^{n} (\text{sin_model}(x_i, \theta_1, \theta_2)-y_i)^2
= \frac{1}{n}\sum_{i=1}^{n}(\theta_1x_i+sin(\theta_2x_i)-y_i)^2$

$\frac{\partial L }{\partial \theta_1}$: the partial derivative of $L$ with respect to $\theta_1$

$\frac{\partial L}{\partial \theta_1}
= \frac{1}{n}\sum_{i=1}^{n}2 \cdot (\theta_1x_i+sin(\theta_2x_i)-y_i) \cdot x_i
$

$\frac{\partial L }{\partial \theta_2}$: the partial derivative of $L$ with respect to $\theta_2, $
  
$\frac{\partial L}{\partial \theta_2}
= \frac{1}{n}\sum_{i=1}^{n}2 \cdot (\theta_1x_i+sin(\theta_2x_i)-y_i) \cdot cos(\theta_2 x_i)  \cdot x_i
 $

Recall that $L(\textbf{x}, \textbf{y}, \theta_1, \theta_2) = \frac{1}{n} \sum_{i=1}^{n} (\textbf{y}_i - \hat{\textbf{y}}_i)^2$



**Complete the functions in the code cell below**  

Specifically, the functions `sin_MSE`, `sin_MSE_dt1` and `sin_MSE_dt2` should compute $L$, $\frac{\partial L }{\partial \theta_1}$ and $\frac{\partial L }{\partial \theta_2}$ respectively. Use the expressions you wrote for $\frac{\partial L }{\partial \theta_1}$ and $\frac{\partial L }{\partial \theta_2}$ to implement these functions. In the functions below, the parameter `theta` is a vector that looks like $\begin{bmatrix} \theta_1 \\ \theta_2 \end{bmatrix}$. We have completed `sin_MSE_gradient`, which calls `dt1` and `dt2` and returns the gradient `dt` for you.

Notes:
* Keep in mind that we are still working with our original set of data, `df`.
* To keep your code a bit more concise, be aware that `np.mean` does the same thing as `np.sum` divided by the length of the numpy array.
* Another way to keep your code more concise is to use the function `sin_model` we defined which computes the output of the model.

In [ ]:
# This function is already implemented for you.
def sin_MSE(theta, x, y):
    """
    Compute the numerical value of the l2 loss of our sinusoidal model given theta

    Keyword arguments:
    theta -- the vector of values theta
    x     -- the vector of x values
    y     -- the vector of y values
    """
    return np.mean((sin_model(x,theta)-y)**2)

# Complete this function.
def sin_MSE_dt1(theta, x, y):
    """
    Compute the numerical value of the partial of l2 loss with respect to theta_1

    Keyword arguments:
    theta -- the vector of values theta
    x     -- the vector of x values
    y     -- the vector of y values
    """
    # add your code here

    return ...



# Complete this function.
def sin_MSE_dt2(theta, x, y):
    """
    Compute the numerical value of the partial of l2 loss with respect to theta_2

    Keyword arguments:
    theta -- the vector of values theta
    x     -- the vector of x values
    y     -- the vector of y values
    """
    # add your code here

    return ...

# This function calls dt1 and dt2 and returns the gradient dt. It is already implemented for you.
def sin_MSE_gradient(theta, x, y):
    """
    Returns the gradient of l2 loss with respect to vector theta

    Keyword arguments:
    theta -- the vector of values theta
    x     -- the vector of x values
    y     -- the vector of y values
    """
    return np.array([sin_MSE_dt1(theta, x, y), sin_MSE_dt2(theta, x, y)])

Use the following code to verify if your implementation is correct.

In [ ]:
# Use synthetic data to verify the correctness of sin_MSE_dt1(theta, x, y)
theta = np.array([1, 1])
x = np.array([1, 2, 3])
y = np.array([1, 2, 3])

# A correct implementation should give a result around 2.056
print(sin_MSE_dt1(theta, x, y))

# A correct implementation should give a result around -0.481
print(sin_MSE_dt2(theta, x, y))

Let's now implement gradient descent.


In [ ]:
def init_theta():
    """Creates an initial theta [0, 0] of shape (2,) as a starting point for gradient descent"""
    return np.array([0, 0])

def grad_desc(loss_f, gradient_loss_f, theta, data, num_iter=20, alpha=0.1):
    """
    Run gradient descent update for a finite number of iterations and static learning rate

    Keyword arguments:
    loss_f -- the loss function to be minimized (used for computing loss_history)
    gradient_loss_f -- the gradient of the loss function to be minimized
    theta -- the vector of values theta to use at first iteration
    data -- the data used in the model
    num_iter -- the max number of iterations
    alpha -- the learning rate (also called the step size)

    Return:
    theta -- the optimal value of theta after num_iter of gradient descent
    theta_history -- the series of theta values over each iteration of gradient descent
    loss_history -- the series of loss values over each iteration of gradient descent
    """
    theta_history = []
    loss_history = []
    for i in range(num_iter):
        theta_history.append(theta)
        loss_history.append(loss_f(theta, data['x'], data['y']))
        d_b = gradient_loss_f(theta, data['x'], data['y'])
        theta = theta - alpha * d_b

    return theta, theta_history, loss_history

In [ ]:
# Initialize the parameters theta
theta_start = init_theta()

# Run gradient descent to update theta
theta_hat, thetas_used, losses_calculated = grad_desc(
    sin_MSE, sin_MSE_gradient, theta_start, df, num_iter=20, alpha=0.01
)

# Print the theta and loss in each iteration
for b, l in zip(thetas_used, losses_calculated):
    print(f"theta: {b}, Loss: {l}")

**Insert code below to create a 2D plot, where the x axis is the iteration number ,and the y axis is the loss in each iteration**

**Insert code below, try each of the learning rate of [0.001, 0.002, 0.005, 0.01], initialize the parameters theta, run the gradient descent to update theta with the specific learning rate and max iteration of 30, and show the plot loss v.s. iteration.**

**Based on the plot, give your observation of the effect of learning rate**


**Is it always better to use a larger learning rate? why? (Hint: Repeat the same plot above with learning rate 0.5 and observe the results before answering this question.)**

Let's visually inspect our results of running gradient descent to optimize $\boldsymbol\theta$.

The code below plots our $x$-values with our model's predicted $\hat{y}$-values over the original scatter plot. You should notice that gradient descent successfully optimized $\boldsymbol\theta$.

In [ ]:
theta_init = init_theta()

theta_est, thetas, loss = grad_desc(sin_MSE, sin_MSE_gradient, theta_init, df)

Plotting our model output over our observaitons shows that gradient descent did  a great job finding both the overall increase (slope) of the data, as well as the oscillation frequency.

In [ ]:
x, y = df['x'], df['y']
y_pred = sin_model(x, theta_est)
plt.plot(x, y_pred, label='Model ($\hat{y}$)')
plt.scatter(x, y, alpha=0.5, label='Observation ($y$)', color='gold')
plt.legend();

Lab 8 is now complete.  Make sure all cells are visible and have been run (rerun if necessary).

The code below converts the ipynb file to PDF, and saves it to where this .ipynb file is. 

In [ ]:
NOTEBOOK_PATH = # Enter here, the path to your notebook file, e.g. "/content/drive/MyDrive/ECEN250/ECEN250_Lab8.ipynb". Do not change the lines below, and make sure you do not have multiple notebooks with the same path.
! pip install playwright
! jupyter nbconvert --to webpdf --allow-chromium-download "$NOTEBOOK_PATH"

Download your notebook as an .ipynb file, then upload it along with the PDF file (saved in the same Google Drive folder as this notebook) to Canvas for Lab 8. Make sure that the PDF file matches your .ipynb file.